# Notebook 00: Limpieza y Análisis Exploratorio de Datos (EDA) - Sofia

Este cuaderno consolida la **auditoría de calidad**, **normalización** y el **análisis exploratorio estadístico y clínico** de las fuentes primarias del proyecto Sofia:
1. **Dra. Gabriela Jufe:** *Psicofarmacología Práctica* (monografías, posologías y efectos adversos).
2. **Dr. Stephen Stahl:** *Prescriber's Guide* (perfil de receptores, interacciones CYP450 y perlas clínicas).
3. **Guías CANMAT e ISBD 2018:** Algoritmos de tratamiento y niveles de evidencia para Trastorno Depresivo Mayor (TDM) y Trastorno Bipolar.

### Objetivos del Cuaderno:
- Auditar la integridad relacional de los datasets en `data/processed/` (nulos, duplicados y validación regex de códigos ATC).
- Visualizar la distribución de familias terapéuticas y perfiles de eventos adversos.
- Construir la matriz de calor (*heatmap*) de interacciones enzimáticas del citocromo P450 (CYP450).
- Evaluar la jerarquía de evidencia de las guías CANMAT para fundamentar la lógica de recuperación de Sofia.


In [1]:
# 1. Configuración de Entorno y Rutas Dinámicas Portables
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
# Detección dinámica y portable de la raíz del proyecto
current_dir = Path.cwd()
project_root = current_dir if (current_dir / 'data').exists() else current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DATA_PROCESSED = project_root / 'data' / 'processed'

# Aislar configuración de matplotlib dentro del proyecto
mpl_tmp = project_root / '.matplotlib_tmp'
mpl_tmp.mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(mpl_tmp)

import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética de gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['figure.dpi'] = 120

print('Entorno de EDA configurado correctamente.')
print(f'- Directorio de datos procesados: {DATA_PROCESSED.relative_to(project_root)}')


Entorno de EDA configurado correctamente.
- Directorio de datos procesados: data/processed


## 1. Carga de Datasets y Auditoría de Integridad

Cargamos los seis datasets clínicos estructurados y comprobamos que no presenten valores nulos, duplicados ni códigos ATC con formato inválido.

In [2]:
import re

# Carga de datasets limpios
df_jufe = pd.read_csv(DATA_PROCESSED / 'jufe_farmacos_clean.csv')
df_jufe_efectos = pd.read_csv(DATA_PROCESSED / 'jufe_efectos_adversos_clean.csv')
df_stahl = pd.read_csv(DATA_PROCESSED / 'stahl_farmacos_clean.csv')
df_stahl_eventos = pd.read_csv(DATA_PROCESSED / 'stahl_eventos_adversos_clean.csv')
df_canmat = pd.read_csv(DATA_PROCESSED / 'canmat_recomendaciones_clean.csv')
df_canmat_alertas = pd.read_csv(DATA_PROCESSED / 'canmat_alertas_clinicas_clean.csv')

atc_regex = re.compile(r'^[A-Z]\d{2}[A-Z]{2}\d{2}$')

def validar_atc(series):
    val = series.dropna()
    if len(val) == 0:
        return 'N/A'
    return '✓ Válido' if val.apply(lambda x: bool(atc_regex.match(str(x)))).all() else '✗ Inválido'

auditoria = [
    {'Dataset': 'Jufe - Fármacos', 'Filas': len(df_jufe), 'Columnas': len(df_jufe.columns), 'Nulos': df_jufe.isnull().sum().sum(), 'Duplicados': df_jufe.duplicated().sum(), 'ATC': validar_atc(df_jufe['atc_codigo'])},
    {'Dataset': 'Jufe - Efectos Adversos', 'Filas': len(df_jufe_efectos), 'Columnas': len(df_jufe_efectos.columns), 'Nulos': df_jufe_efectos.isnull().sum().sum(), 'Duplicados': df_jufe_efectos.duplicated().sum(), 'ATC': 'N/A'},
    {'Dataset': 'Stahl - Fármacos y CYP', 'Filas': len(df_stahl), 'Columnas': len(df_stahl.columns), 'Nulos': df_stahl.isnull().sum().sum(), 'Duplicados': df_stahl.duplicated().sum(), 'ATC': validar_atc(df_stahl['atc_codigo'])},
    {'Dataset': 'Stahl - Eventos Adversos', 'Filas': len(df_stahl_eventos), 'Columnas': len(df_stahl_eventos.columns), 'Nulos': df_stahl_eventos.isnull().sum().sum(), 'Duplicados': df_stahl_eventos.duplicated().sum(), 'ATC': 'N/A'},
    {'Dataset': 'CANMAT - Recomendaciones', 'Filas': len(df_canmat), 'Columnas': len(df_canmat.columns), 'Nulos': df_canmat.isnull().sum().sum(), 'Duplicados': df_canmat.duplicated().sum(), 'ATC': validar_atc(df_canmat['atc'])},
    {'Dataset': 'CANMAT - Alertas Críticas', 'Filas': len(df_canmat_alertas), 'Columnas': len(df_canmat_alertas.columns), 'Nulos': df_canmat_alertas.isnull().sum().sum(), 'Duplicados': df_canmat_alertas.duplicated().sum(), 'ATC': 'N/A'},
]

df_auditoria = pd.DataFrame(auditoria)
print('=== TABLA DE CONTROL DE CALIDAD Y AUDITORÍA DE DATOS ===')
df_auditoria


=== TABLA DE CONTROL DE CALIDAD Y AUDITORÍA DE DATOS ===


## 2. Análisis de Distribución Farmacológica y Códigos ATC de la OMS

Examinamos la cobertura de principios activos en los catálogos principales y su correspondencia con la codificación internacional ATC (Anatomical Therapeutic Chemical Classification).

In [3]:
# Consolidar catálogo único de fármacos con sus códigos ATC
catalogo_farmacos = pd.concat([
    df_jufe[['farmaco', 'atc_codigo', 'clase']].assign(fuente='Jufe'),
    df_stahl[['farmaco', 'atc_codigo', 'clase']].assign(fuente='Stahl')
]).drop_duplicates(subset=['farmaco', 'atc_codigo'])

print(f'Total de fármacos psiquiátricos indexados en el catálogo base: {len(catalogo_farmacos)}')
catalogo_farmacos


Total de fármacos psiquiátricos indexados en el catálogo base: 10


## 3. Visualizaciones Clínicas

### A. Perfil de Seguridad: Distribución de Eventos Adversos por Nivel de Severidad
La farmacovigilancia activa exige distinguir entre efectos leves/moderados de manejo sintomático y eventos graves con riesgo vital que requieren suspensión inmediata o monitorización hematológica/renal.

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Distribución de Severidad en Stahl (paleta dinámica según número de categorías)
conteo_severidad_stahl = df_stahl_eventos['severidad'].value_counts()
colors_stahl = sns.color_palette('coolwarm', len(conteo_severidad_stahl))
axes[0].bar(conteo_severidad_stahl.index, conteo_severidad_stahl.values, color=colors_stahl, width=0.5, edgecolor='black', alpha=0.85)
axes[0].set_title('Eventos Adversos por Severidad (Stahl - Prescriber\'s Guide)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Número de Eventos Registrados', fontsize=10)
for i, v in enumerate(conteo_severidad_stahl.values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold', fontsize=11)
axes[0].set_ylim(0, max(conteo_severidad_stahl.values) + 3)

# 2. Conteo de Efectos Adversos por Fármaco en Jufe
conteo_jufe = df_jufe_efectos['farmaco'].value_counts()
palette_jufe = sns.color_palette('Blues_r', len(conteo_jufe))
axes[1].barh(conteo_jufe.index, conteo_jufe.values, color=palette_jufe, edgecolor='black', alpha=0.85)
axes[1].set_title('Densidad de Efectos Adversos Documentados (Jufe)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Cantidad de Efectos Secundarios', fontsize=10)
for i, v in enumerate(conteo_jufe.values):
    axes[1].text(v + 0.2, i, str(v), va='center', fontweight='bold', fontsize=10)
axes[1].set_xlim(0, max(conteo_jufe.values) + 2)

plt.tight_layout()
plt.show()


### B. Matriz de Calor (Heatmap) de Interacciones con Citocromos CYP450

El sistema enzimático hepático CYP450 es el epicentro de las interacciones farmacocinéticas en psiquiatría. Este mapa de calor cruza la condición de **sustrato** e **inhibidor** en las isoenzimas clave (CYP1A2, CYP2D6, CYP3A4).

In [5]:
# Construir matriz binaria de citocromos a partir del dataset de Stahl
cols_cyp = ['inhibe_cyp2d6', 'inhibe_cyp3a4', 'sustrato_cyp1a2', 'sustrato_cyp2d6', 'sustrato_cyp3a4']
nombres_legibles = ['Inhibe CYP2D6', 'Inhibe CYP3A4', 'Sustrato CYP1A2', 'Sustrato CYP2D6', 'Sustrato CYP3A4']

matriz_cyp = df_stahl.set_index('farmaco')[cols_cyp]
matriz_cyp.columns = nombres_legibles

plt.figure(figsize=(9, 4.5))
sns.heatmap(
    matriz_cyp,
    annot=True,
    cmap='YlOrRd',
    cbar=False,
    linewidths=1.5,
    linecolor='white',
    annot_kws={'size': 12, 'weight': 'bold'}
)
plt.title('Matriz de Interacción Enzimática CYP450 (1 = Actividad / 0 = Neutro)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Eje Metabólico Enzimático', fontsize=11, fontweight='bold')
plt.ylabel('Fármaco', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


### C. Análisis de las Guías Clínicas CANMAT e ISBD 2018

Las guías CANMAT jerarquizan las recomendaciones terapéuticas en base a calidad metodológica estricta (Level 1 a 4). A continuación, analizamos la distribución de estrategias de primera línea, segunda línea y tratamientos no recomendados para Trastorno Depresivo Mayor (TDM) y Trastorno Bipolar.

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Distribución por Línea de Tratamiento y Patología
ct_lineas = pd.crosstab(df_canmat['patologia'], df_canmat['linea'])
ct_lineas.plot(kind='bar', stacked=True, ax=axes[0], colormap='Spectral', edgecolor='black', alpha=0.85)
axes[0].set_title('Recomendaciones CANMAT por Patología y Línea', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Número de Opciones Validadas', fontsize=10)
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Línea de Tratamiento', loc='upper right')

# 2. Distribución de Niveles de Evidencia (paleta dinámica)
conteo_evidencia = df_canmat['nivel_evidencia'].value_counts()
colores_ev = sns.color_palette('Set2', len(conteo_evidencia))
axes[1].pie(
    conteo_evidencia.values,
    labels=conteo_evidencia.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=colores_ev,
    wedgeprops={'edgecolor': 'black', 'linewidth': 1.2},
    textprops={'fontsize': 11, 'weight': 'bold'}
)
axes[1].set_title('Proporción de Niveles de Evidencia CANMAT', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


In [7]:
# Cálculo dinámico de métricas clínicas para el informe
total_rec = len(df_canmat)
conteo_ev = df_canmat['nivel_evidencia'].value_counts()
pct_level1 = (conteo_ev.get('Level 1', 0) / total_rec) * 100

print(f'=== SÍNTESIS DE EVIDENCIA CANMAT (Total: {total_rec} recomendaciones) ===')
for nivel, count in conteo_ev.items():
    print(f'- {nivel}: {count}/{total_rec} ({count/total_rec*100:.1f}%)')


=== SÍNTESIS DE EVIDENCIA CANMAT (Total: 22 recomendaciones) ===
- Level 1: 17/22 (77.3%)
- Level 2: 4/22 (18.2%)
- Level 1 (Daño): 1/22 (4.5%)


## 4. Conclusiones y Repercusiones Clínicas para el Motor RAG Sofia

1. **Zero Hallucination en Jerarquía CANMAT:** Las recomendaciones con mayor respaldo corresponden a **Level 1** (ensayos clínicos aleatorizados controlados doble ciego y metaanálisis), según el cómputo dinámico precedente. Sofia debe priorizar invariablemente estas opciones frente a estrategias de menor respaldo o no recomendadas.
2. **Detección Determinista de Bloqueos CYP2D6:** Fármacos de uso masivo como Paroxetina y Bupropión son potentes inhibidores de CYP2D6, lo que incrementa marcadamente las concentraciones plasmáticas de sustratos como Risperidona o betabloqueantes. Esta interacción debe disparar una alerta crítica de seguridad.
3. **Alerta Roja de Seguridad:** Los antidepresivos ISRS en monoterapia para Depresión Bipolar están clasificados formalmente con evidencia de **Level 1 (Daño)** debido al riesgo severo de viraje maníaco y desestabilización a ciclos rápidos.
4. **Monitoreo de Toxicidad Grave:** Los estabilizadores (Litio y Lamotrigina) y antipsicóticos específicos (Clozapina) presentan eventos adversos potencialmente letales (intoxicación por litemia, síndrome de Stevens-Johnson, agranulocitosis) que requieren protocolos obligatorios de farmacovigilancia.
